<div style='display:flex; align-items:center; justify-content:space-between; border-bottom: 3px solid rgb(255,106,0); padding-bottom:1em; margin-bottom:1em'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 12 &mdash; Variational Autoencoders</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; July 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

In [ ]:
import torch
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from models import AE, VAE
from training import train_ae, train_vae
from helpers import (
    plot_data_examples,
    plot_losses,
    plot_vae_losses,
    plot_examples_with_codes,
    plot_latent_scatter_ae,
    plot_latent_scatter_vae,
    plot_samples_grid,
    plot_interpolation,
    plot_pixel_histogram,
    plot_reconstructions,
)

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>MNIST data</strong>
</div>

28x28 grayscale handwritten digits, pixel values normalized to [0,1]

In [ ]:
# Download MNIST, pixels in [0,1]
transform = transforms.ToTensor()
train_dataset = torchvision.datasets.MNIST(root='../../data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='../../data', train=False, download=True, transform=transform)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")

In [ ]:
# display examples
plot_data_examples(train_dataset)

In [ ]:
# data loaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Plain autoencoder baseline</strong>
</div>

Small CNN encoder/decoder, 2D bottleneck, L2 reconstruction loss

In [ ]:
# basic setup
model_ae = AE(latent_dim=2).to(device)
optimizer = optim.Adam(model_ae.parameters(), lr=1e-3)
num_epochs = 30

In [ ]:
# train model
model_ae, ae_train_losses = train_ae(model_ae, train_loader, optimizer, num_epochs, device)

In [ ]:
# plot loss
plot_losses(ae_train_losses)

What is this '2D latent code' we keep talking about? Just two numbers per image:

In [ ]:
# a few examples with their learned 2D code
plot_examples_with_codes(model_ae, test_dataset, device)

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>The latent space problem</strong>
</div>

Where do all the test images end up in the 2D latent space?

In [ ]:
# scatter of latent codes, colored by digit
plot_latent_scatter_ae(model_ae, test_dataset, device)

The space is lumpy, with gaps and stray regions. What happens if we just pick a random point and decode it?

In [ ]:
# sample z ~ N(0,I) and decode -- this is what 'generating' would mean
plot_samples_grid(model_ae, device)

Not digits. The autoencoder was never asked to make its latent space *fillable* -- it only had to reconstruct the training images, so it crammed them in however was convenient.

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>From autoencoder to variational autoencoder</strong>
</div>

To be able to sample meaningfully, we need the latent space to look like something we know how to sample from -- e.g. a standard normal $\mathcal{N}(0,I)$ -- and to vary smoothly so that nearby points decode to similar images.

The variational autoencoder changes two things:

- the encoder no longer outputs a single point $z$, but a distribution $q(z\mid x) = \mathcal{N}(\mu(x), \sigma^2(x))$
- the loss gets an extra term, $D_{KL}(q(z\mid x) \,\|\, \mathcal{N}(0,I))$, pulling every image's posterior toward the standard normal prior

To keep gradients flowing through the sampling step, we use the **reparameterization trick**: instead of sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ directly, we sample $\epsilon \sim \mathcal{N}(0,I)$ and compute $z = \mu + \sigma \odot \epsilon$. Now the randomness sits in $\epsilon$, and $\mu, \sigma$ remain differentiable.

The full training objective (the negative ELBO) is:

$$\mathcal{L} = \underbrace{-\mathbb{E}_{q(z|x)}\big[\log p(x\mid z)\big]}_{\text{reconstruction}} + \underbrace{D_{KL}\big(q(z\mid x)\,\|\,\mathcal{N}(0,I)\big)}_{\text{regularization}}

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Training the VAE</strong>
</div>

Same CNN backbone as the AE, but the encoder now outputs $\mu$ and $\log\sigma^2$. Reconstruction loss stays L2 for now.

In [ ]:
# basic setup
model_vae = VAE(latent_dim=2).to(device)
optimizer = optim.Adam(model_vae.parameters(), lr=1e-3)
num_epochs = 30

In [ ]:
# train model (L2 reconstruction + KL)
model_vae, vae_train_losses, vae_recon_losses, vae_kl_losses = train_vae(
    model_vae, train_loader, optimizer, num_epochs, device, recon_loss='l2'
)

In [ ]:
# plot the two loss components separately
plot_vae_losses(vae_recon_losses, vae_kl_losses)

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Latent space of the VAE</strong>
</div>

Same scatter plot as before -- scroll back up to compare with the plain autoencoder

In [ ]:
# scatter of latent means, colored by digit
plot_latent_scatter_vae(model_vae, test_dataset, device)

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Generating new digits</strong>
</div>

Now we sample $z \sim \mathcal{N}(0,I)$ again, on the model that was actually trained to make this space meaningful

In [ ]:
# sample and decode
plot_samples_grid(model_vae, device)

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Latent space interpolation</strong>
</div>

Pick two test digits, linearly interpolate between their latent codes, decode the path

In [ ]:
# interpolate between two digits
plot_interpolation(model_vae, test_dataset, device, idx1=0, idx2=1, is_vae=True)

<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Wait a second...</strong>
</div>

We assumed a Gaussian likelihood (L2 loss) for the pixels without really checking it. What does the pixel value distribution actually look like?

In [ ]:
# pixel value histogram
plot_pixel_histogram(train_dataset)

Pixels pile up near 0 and near 1 -- 'ink' or 'no ink'. That is much closer to a **Bernoulli** distribution than a Gaussian one. The reconstruction loss is really just $-\log p_\theta(x\mid z)$, and the likelihood we assume determines the loss:

- Gaussian likelihood $\Rightarrow$ L2 loss
- Bernoulli likelihood $\Rightarrow$ binary cross-entropy (BCE) loss

Let's retrain the same architecture with BCE instead, and see whether it helps.

In [ ]:
# basic setup -- identical architecture, only the loss changes
model_vae_bce = VAE(latent_dim=2).to(device)
optimizer = optim.Adam(model_vae_bce.parameters(), lr=1e-3)
num_epochs = 30

In [ ]:
# train model (BCE reconstruction + KL)
model_vae_bce, bce_train_losses, bce_recon_losses, bce_kl_losses = train_vae(
    model_vae_bce, train_loader, optimizer, num_epochs, device, recon_loss='bce'
)

In [ ]:
# plot the two loss components
plot_vae_losses(bce_recon_losses, bce_kl_losses)

Compare generated digits with the L2 version above:

In [ ]:
# sample and decode -- compare with the L2-VAE samples from the 'Generating new digits' section
plot_samples_grid(model_vae_bce, device)

And reconstructions side by side: L2-VAE (top) vs BCE-VAE (bottom) for the same test images

In [ ]:
# reconstruction comparison
example_indices = [0, 1, 2, 3, 4]
plot_reconstructions(model_vae, test_dataset, device, example_indices, is_vae=True)
plot_reconstructions(model_vae_bce, test_dataset, device, example_indices, is_vae=True)